# 04 — Faster R-CNN Training (BDD100K)

Train a Faster R-CNN model with ResNet-50 FPN backbone on the BDD100K subset.
This notebook handles dataset loading, model setup, training with early stopping, and saving outputs.

In [ ]:
import shutil
import os

source_dir = '/kaggle/input/datasets/bddk100k-object-detection/Object-Detection'
destination_dir = '/kaggle/working/bdd100k_project'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print("Files copied to /kaggle/working/src and ready to use!")
else:
    print("Source directory not found. Check your paths!")

In [ ]:
import sys
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import json
import os
import shutil

project_path = "/kaggle/working/bdd100k_project"
if project_path not in sys.path:
    sys.path.append(project_path)
    sys.path.append(os.path.join(project_path, 'src'))

from src.utils import SEED, CLASS_MAP, FRCNN_CLASS_MAP, NUM_CLASSES, seed_everything, log_environment
from src.fasterrcnn_dataset import BDD100KDataset
from src.fasterrcnn_utils import collate_fn, build_fasterrcnn, train_one_epoch, val_one_epoch

## 1. Environment & Seed

In [ ]:
seed_everything(SEED)
log_environment()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"FRCNN_CLASS_MAP: {FRCNN_CLASS_MAP}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

## 2. Dataset & DataLoaders

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/yolo-subset"

TRAIN_IMG_DIR = f"{DATASET_ROOT}/bdd100k-yolo-subset-v1/images/train"
VAL_IMG_DIR = f"{DATASET_ROOT}/bdd100k-yolo-subset-v1/images/val"

train_dataset = BDD100KDataset(
    image_dir=TRAIN_IMG_DIR,
    annotation_file=f"{DATASET_ROOT}/train_annotations.json",
    class_map=FRCNN_CLASS_MAP,
)

val_dataset = BDD100KDataset(
    image_dir=VAL_IMG_DIR,
    annotation_file=f"{DATASET_ROOT}/val_annotations.json",
    class_map=FRCNN_CLASS_MAP,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
BATCH_SIZE = 6

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 3. Model Setup

In [ ]:
model = build_fasterrcnn(num_classes=NUM_CLASSES)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in params):,}")

## 4. Optimizer & Scheduler

In [ ]:
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Optimizer: SGD (lr=0.005, momentum=0.9, weight_decay=0.0005)")
print("Scheduler: StepLR (step_size=5, gamma=0.1)")

## 5. Training Loop

In [ ]:
NUM_EPOCHS = 15
PATIENCE = 5

best_val_loss = float("inf")
epochs_no_improve = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "loss_classifier": [],
    "loss_box_reg": [],
    "loss_objectness": [],
    "loss_rpn_box_reg": [],
}

In [ ]:
import time

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, loss_components = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss = val_one_epoch(model, val_loader, device, epoch)
    lr_scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    for key in ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]:
        history[key].append(loss_components.get(key, 0.0))

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/fasterrcnn_best.pth")
        print(f"  -> Best model saved (val_loss={val_loss:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

train_time = time.time() - train_start
print(f"\nTotal training time: {train_time / 3600:.2f} hours")

## 6. Loss Curve

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("/kaggle/working/results/plots", exist_ok=True)

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN — Training & Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/results/plots/fasterrcnn_loss_curve.png", dpi=150)
plt.show()

## 7. Save Outputs

In [ ]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

shutil.copy("/kaggle/working/fasterrcnn_best.pth",
            "/kaggle/working/outputs/fasterrcnn_bdd100k_best.pth")

history["training_time_hours"] = train_time / 3600
with open("/kaggle/working/outputs/fasterrcnn_loss_history.json", "w") as f:
    json.dump(history, f, indent=2)

shutil.copy("/kaggle/working/results/plots/fasterrcnn_loss_curve.png",
            "/kaggle/working/outputs/fasterrcnn_loss_curve.png")

print("All outputs saved to /kaggle/working/outputs/")

In [ ]:
import pandas as pd

os.makedirs("/kaggle/working/results/metrics", exist_ok=True)

df = pd.DataFrame(history)
df.index.name = "epoch"
df.index += 1
df.to_csv("/kaggle/working/results/metrics/fasterrcnn_results.csv")
print("Loss history saved to results/metrics/fasterrcnn_results.csv")
df